# Protein Interaction NN Development Plan

## Objective
Build a new model that predicts **protein-protein interaction likelihood** from two variable-size vertex sets:
- Query protein vertices (interface-focused subset or full set)
- Matched protein vertices (full set or sampled set)

Each vertex includes:
- MaSIF descriptor vector (`D`, verify whether current run exports 16D or 80D)
- 3D coordinates (`x, y, z`)
- Optional geometric extras (normal, curvature, atom-distance prior)

The new model will extend/replace `AlignmentEvaluationNN` with:
1. End-to-end set-to-set interaction scoring
2. Built-in geometric + descriptor reasoning
3. Permutation invariance for variable-length inputs
4. Interpretable outputs (correspondences, transform quality, per-vertex importance)

## Delivery Strategy
Implement in three stages so we get a usable model early, then add geometry-aware refinement:
- **V1 (baseline):** Set encoder + cross-attention + protein-level classifier
- **V2 (core):** Add differentiable correspondence and rigid transform refinement
- **V3 (optional):** Upgrade encoder to equivariant message passing (EGNN/SE(3)-Transformer)

---


## 1) Data Definition and Sampling Plan

### 1.1 Training examples
Each training sample is a pair `(Q, M, y)`:
- `Q`: query vertices (shape `[N, F]`), from known/putative interface of chain p1
- `M`: matched vertices (shape `[M, F]`), from chain p2 (all vertices or large subset)
- `y`: interaction label (`1` for true interacting pair, `0` for negative pair)

`F` includes descriptor channels + xyz (+ optional normals/curvature).

### 1.2 Positive pairs
From known complexes:
- Use query interface vertices from p1 (`N` variable)
- Use all (or sampled) vertices from interacting p2 (`M` variable)
- Keep chain-level identity metadata for split control

### 1.3 Negative pairs (hard-negative heavy)
Use mixed negatives to avoid trivial separation:
1. **Cross-complex random negatives** (different PDB)
2. **Same-fold / descriptor-similar negatives**
3. **Within-complex incorrect partners/regions**
4. **Geometric decoys** (descriptor-similar but poor post-transform geometry)

Target ratio (starting point): `1 : 3` (positive : negative), with at least 50% hard negatives.

### 1.4 Split policy (leakage prevention)
- Split by complex/family, not by patch
- Ensure no chain-level leakage between train/val/test
- Keep an external benchmark split for final report

### 1.5 Batching variable-size sets
For each mini-batch:
- pad query to `N_max`, matched to `M_max`
- create masks `mask_q`, `mask_m`
- optional random subsampling for very large proteins (e.g., farthest-point or interface-biased)


## 2) Model Architecture (Concrete)

### 2.1 V1 Baseline (fastest path)
**Input**
- `Q_desc: [B, N, D]`, `Q_xyz: [B, N, 3]`
- `M_desc: [B, M, D]`, `M_xyz: [B, M, 3]`
- masks for `N` and `M`

**Blocks**
1. Shared point encoder (MLP/1x1 Conv):
   - `h_q = Enc([desc, xyz])`, `h_m = Enc([desc, xyz])`
2. Cross-attention (query <-> matched) with masking
3. Set pooling (attention pooling + max/mean concat)
4. MLP head -> interaction probability `p_int`

**Properties**
- Permutation-invariant over each set
- Handles variable `N, M`
- Joint descriptor + geometric input

### 2.2 V2 Core Upgrade (recommended target)
Add differentiable geometry refinement inside network:
1. Pairwise affinity logits `S_ij` from cross features
2. Soft correspondence matrix `P = softmax(S)` with masks
3. Weighted rigid alignment (differentiable Kabsch) -> `(R, t)`
4. Transform matched coordinates: `M' = R M + t`
5. Recompute cross features using transformed geometry
6. Final pooled representation -> `p_int`

This lets the model learn the hypothesis directly: good binders can be transformed so descriptor-similar points become geometrically close.

### 2.3 V3 Optional Upgrade
Replace basic encoder with EGNN/SE(3)-aware message passing for stronger geometric reasoning, while retaining V2 heads and losses.


## 3) Losses, Objectives, and Outputs

### 3.1 Primary objective
- `L_cls`: binary cross-entropy on protein-level interaction label `y`

### 3.2 Auxiliary objectives (V2+)
- `L_geo`: correspondence-weighted geometric consistency after transform
  - encourage small `||q_i - m'_j||` where `P_ij` is high
- `L_desc`: correspondence-weighted descriptor consistency
  - encourage small `||d_q_i - d_m_j||` where `P_ij` is high
- `L_reg`: regularization terms
  - correspondence entropy control
  - transform stability
  - optional clash penalty

Total loss:
`L = w_cls*L_cls + w_geo*L_geo + w_desc*L_desc + w_reg*L_reg`

Start weights: `w_cls=1.0, w_geo=0.3, w_desc=0.3, w_reg=0.05` then tune.

### 3.3 Inference outputs for interpretability
For each protein pair output:
1. Interaction probability `p_int`
2. Top correspondences (`P_ij` peaks)
3. Estimated transform `(R, t)`
4. Post-transform fit metrics
   - mean nearest-neighbor distance
   - descriptor agreement score
5. Per-vertex saliency/importance on query and matched proteins


## 4) Training Workflow and Milestones

### Milestone A: Data readiness
1. Verify descriptor dimensionality in current exported files (`D=16` vs `D=80`)
2. Build pair dataset with strict split control
3. Implement negative mining pipeline and cached indices
4. Add quick diagnostics:
   - set-size distribution (`N`, `M`)
   - descriptor-distance distribution (pos vs neg)
   - geometry-distance distribution after rough alignment

### Milestone B: V1 baseline model
1. Train baseline classifier without transform refinement
2. Metrics:
   - ROC-AUC, PR-AUC
   - calibration (Brier score / reliability curve)
3. Establish benchmark runtime and memory

### Milestone C: V2 differentiable alignment
1. Add soft correspondences + weighted Kabsch
2. Train with multi-task loss
3. Compare to V1 and current AlignmentEvaluationNN pipeline

### Milestone D: Robustness and interpretability
1. Evaluate on hard negatives and family-held-out split
2. Report interpretability artifacts for representative cases
3. Produce failure analysis (false positives/negatives)

### Suggested timeline
- Week 1: Milestone A
- Week 2: Milestone B
- Week 3: Milestone C
- Week 4: Milestone D + ablations


## 5) Evaluation Protocol and Ablation Matrix

### 5.1 Core metrics
- Protein-pair ROC-AUC / PR-AUC
- Top-k recall for interacting pairs
- Ranking quality against decoys
- Calibration and thresholded precision/recall

### 5.2 Geometric quality metrics
- Post-transform RMSD (where ground truth exists)
- Nearest-neighbor geometric fit
- Descriptor agreement at matched correspondences

### 5.3 Ablation plan
1. Descriptors only vs xyz only vs descriptors+xyz
2. No-transform (V1) vs transform-refined (V2)
3. Random negatives only vs mixed hard negatives
4. Interface-only query subset vs full query surface
5. Different pooling choices (mean/max/attention)

### 5.4 Success criteria (initial)
- Improve protein-level PR-AUC over current alignment pipeline
- Better ranking under hard negatives
- Stable calibration across held-out families
- Interpretable correspondences consistent with known interfaces


## 6) Implementation Checklist (Repo-Oriented)

### Immediate next actions
- [ ] Confirm descriptor dimensionality from current `*_desc_*.npy`
- [ ] Define standardized pair-sample format (`npz` or parquet-like index + npy blobs)
- [ ] Implement dataset class with masks and dynamic padding
- [ ] Implement V1 baseline model + trainer
- [ ] Add experiment config file (model/data/loss/hard-negative knobs)
- [ ] Add evaluation script with saved predictions and plots

### Reproducibility
- [ ] Fixed random seeds and deterministic split files
- [ ] Save train/val/test IDs and negative-mining snapshots
- [ ] Version model checkpoints and feature schema

### Risk controls
- If training becomes unstable:
  1. freeze transform head for first warm-up epochs
  2. increase `w_cls`, decrease geometry losses
  3. clip gradients and reduce learning rate

---

## Notes
- This plan intentionally starts simple (V1) to get a strong baseline quickly.
- V2 is the key model to satisfy your biological hypothesis (joint descriptor + geometry after learned rigid alignment).
- Keep the old AlignmentEvaluationNN pipeline as a reference baseline during development for fair comparison.


## 7) Minimal File-by-File Implementation Map

### A. Data and config layer
1. `masif_seed_search/data/protein_interaction_nn/configs/protein_interaction_v1.yaml`
   - Central experiment config: paths, descriptor dim `D`, batch sizes, loss weights, split files, negative-mix ratios.

2. `masif_seed_search/data/protein_interaction_nn/configs/protein_interaction_v2.yaml`
   - Same as V1 + transform/correspondence settings (`w_geo`, `w_desc`, entropy regularization, Kabsch options).

3. `masif_seed_search/source/protein_interaction/build_pair_index.py`
   - Build train/val/test pair index files.
   - Enforce leakage-safe split by complex/family.
   - Output: `pairs_{train,val,test}.csv` (query_id, matched_id, label, metadata).

4. `masif_seed_search/source/protein_interaction/mine_hard_negatives.py`
   - Add hard negatives using descriptor-similarity and decoy heuristics.
   - Output: hard-negative tables or cached `.npy` index arrays.

5. `masif_seed_search/source/protein_interaction/dataset.py`
   - Dataset loader for variable-size sets.
   - Reads descriptors + xyz, applies subsampling, returns masks and padded tensors.

### B. Model layer
6. `masif_seed_search/source/protein_interaction/models/point_encoder.py`
   - Shared point encoder (MLP/1x1-conv) for per-vertex features.

7. `masif_seed_search/source/protein_interaction/models/cross_set_v1.py`
   - V1 baseline: encoder + cross-attention + set pooling + classifier.

8. `masif_seed_search/source/protein_interaction/models/geometry_heads.py`
   - Reusable geometry utilities for V2:
     - affinity -> soft correspondence
     - weighted/differentiable Kabsch
     - optional transform regularization

9. `masif_seed_search/source/protein_interaction/models/cross_set_v2.py`
   - V2 model: V1 backbone + correspondence/transform refinement + multi-task outputs.

### C. Training and evaluation layer
10. `masif_seed_search/source/protein_interaction/losses.py`
    - `L_cls`, `L_geo`, `L_desc`, `L_reg` with weighted sum helper.

11. `masif_seed_search/source/protein_interaction/train.py`
    - Generic trainer for V1/V2 from config.
    - Saves checkpoints, logs, and validation predictions.

12. `masif_seed_search/source/protein_interaction/evaluate.py`
    - Computes ROC-AUC/PR-AUC/calibration and ranking metrics.
    - Exports per-pair outputs (probability, transform, match stats).

13. `masif_seed_search/source/protein_interaction/ablate.py`
    - Runs controlled ablations (desc-only, xyz-only, no-transform, negative policy variants).

### D. Analysis and reproducibility
14. `masif_seed_search/data/protein_interaction_nn/splits/`
    - Versioned split files and family exclusions.

15. `masif_seed_search/data/protein_interaction_nn/analysis/`
    - Saved metrics tables, plots, calibration curves, and failure-case reports.

16. `masif_seed_search/data/protein_interaction_nn/README.md`
    - One-page runbook: setup, data prep, train, eval, ablations, expected artifacts.

---

### Suggested first implementation order (minimal)
1. `build_pair_index.py` -> `dataset.py` -> `cross_set_v1.py` -> `train.py` -> `evaluate.py`
2. Add `mine_hard_negatives.py` and retrain V1 with mixed negatives
3. Implement `geometry_heads.py` + `cross_set_v2.py` + `losses.py`
4. Run `ablate.py` and finalize benchmark comparison vs current AlignmentEvaluationNN
